# Card Reader

Loads the model we trained in Colab (`models/best.onnx`), moves the arm to a camera pose, takes a picture, and reads the cards.

**Before running:** put `best.onnx` from Colab into `HackRPI26/models/` on the Jetson.

In [ ]:
import json
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)
print("Poses:", list(poses))

## Load the model
Uses `onnxruntime` if it's installed, otherwise OpenCV's built-in DNN. Prints the output shape so we know the model loaded right.

In [ ]:
MODEL_PATH = 'models/best.onnx'

try:
    import onnxruntime as ort
    session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    BACKEND = 'onnxruntime'
except ImportError:
    net = cv2.dnn.readNetFromONNX(MODEL_PATH)
    BACKEND = 'opencv'
print("Backend:", BACKEND)

## Detection + blackjack helpers

In [ ]:
CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as training
IMGSZ = 416        # same size we trained/exported with
CONF = 0.5         # ignore detections less confident than this


def letterbox(frame, size=IMGSZ):
    """Resize keeping the shape, pad the rest with gray (same as YOLO training)."""
    h, w = frame.shape[:2]
    r = min(size / h, size / w)
    nw, nh = int(round(w * r)), int(round(h * r))
    pad_x, pad_y = (size - nw) / 2, (size - nh) / 2
    img = cv2.resize(frame, (nw, nh))
    img = cv2.copyMakeBorder(img, int(round(pad_y - 0.1)), int(round(pad_y + 0.1)),
                             int(round(pad_x - 0.1)), int(round(pad_x + 0.1)),
                             cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return img, r, pad_x, pad_y


def run_model(blob):
    if BACKEND == 'onnxruntime':
        return session.run(None, {input_name: blob})[0]
    net.setInput(blob)
    return net.forward()


def detect(frame, conf=CONF):
    """Returns a list of (label, confidence, (x1, y1, x2, y2)) in the original picture's pixels."""
    img, r, pad_x, pad_y = letterbox(frame)
    blob = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    out = np.asarray(run_model(blob))[0]

    boxes, scores, ids = [], [], []
    if out.ndim == 2 and out.shape[1] == 6:
        # YOLO26 end-to-end output: each row = x1 y1 x2 y2 score class (no NMS needed)
        for x1, y1, x2, y2, s, c in out:
            if s >= conf:
                boxes.append([x1, y1, x2, y2]); scores.append(float(s)); ids.append(int(c))
    else:
        # Older YOLO output: (4 + classes) x N, boxes are cx cy w h -> needs NMS
        if out.shape[0] != 4 + len(CLASSES):
            out = out.T
        cls_scores = out[4:]
        ids_all = cls_scores.argmax(0)
        s_all = cls_scores.max(0)
        keep = s_all >= conf
        cx, cy, w, h = out[:4, keep]
        xywh = np.stack([cx - w / 2, cy - h / 2, w, h], 1)
        s_k, id_k = s_all[keep], ids_all[keep]
        for i in np.array(cv2.dnn.NMSBoxes(xywh.tolist(), s_k.tolist(), conf, 0.5)).flatten():
            x, y, bw, bh = xywh[i]
            boxes.append([x, y, x + bw, y + bh]); scores.append(float(s_k[i])); ids.append(int(id_k[i]))

    results = []
    H, W = frame.shape[:2]
    for (x1, y1, x2, y2), s, c in zip(boxes, scores, ids):
        # undo the letterbox: back to original picture pixels
        x1 = int(np.clip((x1 - pad_x) / r, 0, W)); x2 = int(np.clip((x2 - pad_x) / r, 0, W))
        y1 = int(np.clip((y1 - pad_y) / r, 0, H)); y2 = int(np.clip((y2 - pad_y) / r, 0, H))
        results.append((CLASSES[c], s, (x1, y1, x2, y2)))
    return results


def hand_value(labels):
    """Blackjack total. Aces count 11 unless that busts. DOWN cards are ignored."""
    total, aces = 0, 0
    for l in labels:
        if l == 'DOWN':
            continue
        if l == 'A':
            total += 11; aces += 1
        elif l in ('J', 'Q', 'K', '10'):
            total += 10
        else:
            total += int(l)
    while total > 21 and aces:
        total -= 10; aces -= 1
    return total, aces > 0   # (total, is_soft)

In [ ]:
# Quick check: run the model once on a blank picture
t = time.time()
img, _, _, _ = letterbox(np.zeros((480, 640, 3), np.uint8))
blob = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype(np.float32) / 255.0
print("Model output shape:", np.asarray(run_model(blob)).shape, f" ({time.time() - t:.2f} s)")

## Camera + arm helpers

In [ ]:
def move(q):
    # 6th value is the gripper (held at 90), 1500 = take 1.5 seconds
    arm.Arm_serial_servo_write6(*q, 90, 1500)
    time.sleep(2)


def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def show(frame, dets, title=''):
    img = frame.copy()
    for label, s, (x1, y1, x2, y2) in dets:
        color = (128, 128, 128) if label == 'DOWN' else (0, 255, 0)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{label} {s:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


def read_cards(pose_name):
    move(poses[pose_name])
    frame = take_picture()
    if frame is None:
        print("No camera frame")
        return []
    t = time.time()
    dets = detect(frame)
    labels = [d[0] for d in dets]
    total, soft = hand_value(labels)
    print(f"{pose_name}: {labels}  ->  total {total}{' (soft)' if soft else ''}   [{time.time() - t:.2f} s]")
    show(frame, dets, f"{pose_name}: {labels}  total {total}")
    return labels

## Read the dealer's cards

In [ ]:
dealer = read_cards('dealer')
up = [c for c in dealer if c != 'DOWN']
print("Dealer up card:", up, "| face-down cards:", dealer.count('DOWN'))

## Read my cards

In [ ]:
mine = read_cards('mycards')
total, soft = hand_value(mine)
print("My hand:", mine, "->", total, "| Decision:", "HIT" if total < 17 else "STAND")

## Test loop
Stays at one pose. Change the cards, hit Enter to read again. Type anything + Enter to stop.
Good for checking accuracy: count how many reads are right.

In [ ]:
POSE = 'mycards'

move(poses[POSE])
while input("Enter = read cards, anything else = stop: ") == '':
    frame = take_picture()
    if frame is None:
        print("No camera frame")
        continue
    dets = detect(frame)
    labels = [d[0] for d in dets]
    total, soft = hand_value(labels)
    print(labels, "->", total, "(soft)" if soft else "")
    show(frame, dets, f"{labels}  total {total}")

## Done

In [ ]:
move(poses['home'])
cap.release()